# Bài 12: Clone bảng (SHALLOW/DEEP CLONE) & Tổng kết

## Mục tiêu
- Phân biệt `SHALLOW CLONE` và `DEEP CLONE`, biết khi nào dùng loại nào.
- Ôn tập toàn bộ kiến thức Delta Lake qua 1 **mini project** kết hợp nhiều bài trước.
- Có 1 checklist best practices để áp dụng vào dự án thật.


## 12.1. `SHALLOW CLONE` vs `DEEP CLONE`

```sql
CREATE TABLE db.tbl_shallow SHALLOW CLONE db.tbl_source;
CREATE TABLE db.tbl_deep    DEEP CLONE    db.tbl_source;
```

| | **SHALLOW CLONE** | **DEEP CLONE** |
|---|---|---|
| Copy dữ liệu vật lý (file Parquet)? | **Không** — chỉ copy metadata (`_delta_log`), trỏ tới file gốc | **Có** — copy toàn bộ file dữ liệu sang location mới |
| Tốc độ tạo | Rất nhanh (chỉ ghi vài file JSON) | Chậm hơn, tỉ lệ với dung lượng dữ liệu |
| Độc lập với bảng gốc? | **Không hoàn toàn** — nếu bảng gốc bị `VACUUM` xoá file cũ, clone có thể mất khả năng đọc dữ liệu tương ứng | **Có** — hoàn toàn độc lập, an toàn khi bảng gốc bị xoá/VACUUM |
| Use case | Thử nghiệm nhanh (thử schema change, thử `OPTIMIZE`, thử pipeline mới) mà không tốn storage | Backup, tạo snapshot production để dev/staging test, migrate sang catalog/storage khác |

Cả 2 loại clone đều tạo ra 1 bảng Delta **hoàn toàn mới, độc lập về ghi**: ghi/update/delete vào bản clone **không ảnh hưởng** tới bảng gốc (và ngược lại) — vì mỗi bảng có `_delta_log` riêng, các thay đổi sau khi clone chỉ được ghi nhận trong log tương ứng.

Có thể clone tại 1 version cụ thể: `CREATE TABLE db.snap DEEP CLONE db.tbl VERSION AS OF 5;` — kết hợp Time Travel (Bài 5) với Clone để tạo snapshot production tại 1 thời điểm cụ thể.


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai12"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai12-clone")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 12.2. Ví dụ minh hoạ

In [ ]:
spark.sql("DROP TABLE IF EXISTS bai12.source_tbl")
spark.createDataFrame([(1,"A"),(2,"B"),(3,"C")], ["id","name"]).write.format("delta").saveAsTable("bai12.source_tbl")

spark.sql("DROP TABLE IF EXISTS bai12.shallow_copy")
spark.sql("DROP TABLE IF EXISTS bai12.deep_copy")
spark.sql("CREATE TABLE bai12.shallow_copy SHALLOW CLONE bai12.source_tbl")
spark.sql("CREATE TABLE bai12.deep_copy DEEP CLONE bai12.source_tbl")

spark.sql("SELECT * FROM bai12.shallow_copy ORDER BY id").show()
spark.sql("SELECT * FROM bai12.deep_copy ORDER BY id").show()


In [ ]:
# Ghi vao ban clone khong anh huong ban goc
spark.sql("INSERT INTO bai12.deep_copy VALUES (4, 'D-only-in-deep-copy')")
print("source_tbl:", spark.table("bai12.source_tbl").count(), "dong")
print("deep_copy:", spark.table("bai12.deep_copy").count(), "dong")


## 12.3. Mini project — Tổng hợp toàn bộ kiến thức

Xây dựng 1 pipeline nhỏ mô phỏng thực tế, kết hợp hầu hết các kỹ thuật đã học từ Bài 1-11:

1. **(Bài 2, 6)** Tạo bảng `bai12.orders_bronze` (raw) với constraint `qty > 0`, bật `delta.enableChangeDataFeed = true`.
2. **(Bài 3)** Nạp dữ liệu ban đầu bằng `append`, partition theo `order_date`.
3. **(Bài 4)** Dùng `MERGE INTO` để upsert 1 batch "sửa đơn hàng" (một số đơn cập nhật `qty`, một số đơn hoàn toàn mới).
4. **(Bài 9)** Đọc **Change Data Feed** để lấy đúng phần thay đổi vừa rồi.
5. **(Bài 10)** Dùng `foreachBatch` streaming để đẩy các thay đổi đó vào bảng tổng hợp `bai12.orders_summary (order_date DATE, total_qty LONG)`.
6. **(Bài 7, 8)** Chạy `OPTIMIZE ... ZORDER BY` trên bronze, sau đó `VACUUM DRY RUN` để xem file sẽ được dọn.
7. **(Bài 5)** Dùng `DESCRIBE HISTORY` + `VERSION AS OF` để "audit" lại toàn bộ các bước trên.
8. **(Bài 12)** Tạo `DEEP CLONE` của `bai12.orders_bronze` để làm snapshot backup trước khi thử nghiệm tiếp.

Phần code mẫu dưới đây thực hiện các bước 1-3, 6-8; bước 4-5 (CDF + streaming foreachBatch) để lại làm bài tập lớn (Bài 4 phần thực hành) vì đã luyện kỹ ở Bài 9-10.


In [ ]:
# Buoc 1-2: tao bang bronze co constraint + CDF, nap du lieu ban dau
from datetime import date

spark.sql("DROP TABLE IF EXISTS bai12.orders_bronze")
spark.sql("""
CREATE TABLE bai12.orders_bronze (
    order_id INT, product STRING, qty INT, order_date DATE
)
USING DELTA
PARTITIONED BY (order_date)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
spark.sql("ALTER TABLE bai12.orders_bronze ADD CONSTRAINT qty_positive CHECK (qty > 0)")

initial = spark.createDataFrame(
    [(1,"Keyboard",5,date(2024,6,1)), (2,"Mouse",10,date(2024,6,1)), (3,"Monitor",2,date(2024,6,2))],
    ["order_id","product","qty","order_date"],
)
initial.write.format("delta").mode("append").saveAsTable("bai12.orders_bronze")
spark.sql("SELECT * FROM bai12.orders_bronze ORDER BY order_id").show()


In [ ]:
# Buoc 3: MERGE upsert - sua qty don 1, them don moi 4
changes = spark.createDataFrame(
    [(1,"Keyboard",8,date(2024,6,1)), (4,"Webcam",3,date(2024,6,2))],
    ["order_id","product","qty","order_date"],
)
changes.createOrReplaceTempView("orders_changes")

spark.sql("""
MERGE INTO bai12.orders_bronze t USING orders_changes s
ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * FROM bai12.orders_bronze ORDER BY order_id").show()


In [ ]:
# Buoc 6: OPTIMIZE + ZORDER, roi VACUUM DRY RUN (khong xoa that trong mini project nay)
spark.sql("OPTIMIZE bai12.orders_bronze ZORDER BY (product)").show(truncate=False)
spark.sql("VACUUM bai12.orders_bronze DRY RUN").show(truncate=False)


In [ ]:
# Buoc 7: audit lai lich su
spark.sql("DESCRIBE HISTORY bai12.orders_bronze").select("version","operation","operationMetrics").show(truncate=False)


In [ ]:
# Buoc 8: DEEP CLONE lam backup
spark.sql("DROP TABLE IF EXISTS bai12.orders_bronze_backup")
spark.sql("CREATE TABLE bai12.orders_bronze_backup DEEP CLONE bai12.orders_bronze")
print("Backup co", spark.table("bai12.orders_bronze_backup").count(), "dong, doc lap voi bang goc.")


## 12.4. Thực hành

**Bài 1** — Tạo `SHALLOW CLONE` của `bai12.orders_bronze` tên `bai12.orders_bronze_shallow`. Chạy `VACUUM ... RETAIN 0 HOURS` (tắt safety check) trên bảng **gốc** rồi thử đọc `orders_bronze_shallow` — bạn có gặp lỗi không? Giải thích bằng lý thuyết mục 12.1.

**Bài 2** — Hoàn thiện bước 4-5 còn thiếu trong mini project: đọc CDF của `bai12.orders_bronze` (từ version của bước MERGE), rồi dùng `foreachBatch` cập nhật bảng `bai12.orders_summary (order_date DATE, total_qty LONG)` — tổng `qty` theo từng `order_date`, xử lý đúng cả `update_postimage` lẫn `insert` (gợi ý: có thể đơn giản hoá bằng cách group lại theo `order_date` và ghi đè tổng cho các ngày bị ảnh hưởng, dùng `replaceWhere` hoặc `MERGE`).

**Bài 3** — Viết 1 query dùng `VERSION AS OF` để xác nhận: trước khi chạy MERGE ở bước 3, đơn hàng `order_id = 1` có `qty = 5`; sau MERGE, `qty = 8`.

**Bài 4 (tổng hợp)** — Từ đầu, tự thiết kế và xây 1 mini pipeline khác cho chủ đề tự chọn (ví dụ: log ứng dụng, giao dịch ngân hàng, đơn hàng thương mại điện tử). Yêu cầu tối thiểu: có bảng partition hợp lý, có ít nhất 1 `MERGE`, có bật CDF, có chạy `OPTIMIZE`, và có dùng `DESCRIBE HISTORY` để chứng minh pipeline hoạt động đúng qua nhiều version.

**Bài 5 (checklist)** — Tự trả lời (markdown): với bảng bạn build ở Bài 4, bạn sẽ đặt `OPTIMIZE`/`VACUUM` chạy theo lịch nào, retention bao nhiêu, có cần CDF không, có cần Z-order theo cột nào không? Giải thích từng lựa chọn.


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

_Viết checklist của bạn ở đây._

---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1
spark.sql("DROP TABLE IF EXISTS bai12.orders_bronze_shallow")
spark.sql("CREATE TABLE bai12.orders_bronze_shallow SHALLOW CLONE bai12.orders_bronze")

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql("VACUUM bai12.orders_bronze RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

try:
    spark.sql("SELECT * FROM bai12.orders_bronze_shallow ORDER BY order_id").show()
    print("Doc duoc (co the file dang active chua bi vacuum dong cham toi).")
except Exception as e:
    print("Loi:", type(e).__name__, "- vi SHALLOW CLONE khong co ban sao du lieu rieng,")
    print("khi bang goc VACUUM xoa file vat ly, clone mat kha nang doc phan du lieu tuong ung.")


In [ ]:
# Dap an Bai 2
from delta.tables import DeltaTable

spark.sql("DROP TABLE IF EXISTS bai12.orders_summary")
spark.sql("CREATE TABLE bai12.orders_summary (order_date DATE, total_qty LONG) USING DELTA")

# Lay CDF tu toan bo lich su cho don gian trong bai tap; thuc te se dung startingVersion = version cua MERGE
cdf = spark.sql("SELECT * FROM table_changes('bai12.orders_bronze', 0)") \
    .filter("_change_type IN ('insert', 'update_postimage')")

agg = cdf.groupBy("order_date").sum("qty").withColumnRenamed("sum(qty)", "total_qty")

(
    DeltaTable.forName(spark, "bai12.orders_summary").alias("t")
    .merge(agg.alias("s"), "t.order_date = s.order_date")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
spark.sql("SELECT * FROM bai12.orders_summary ORDER BY order_date").show()


In [ ]:
# Dap an Bai 3
hist = spark.sql("DESCRIBE HISTORY bai12.orders_bronze").select("version","operation").orderBy("version").collect()
merge_version = next(r["version"] for r in hist if r["operation"] == "MERGE")

print("Truoc MERGE (version", merge_version - 1, "):")
spark.sql(f"SELECT * FROM bai12.orders_bronze VERSION AS OF {merge_version - 1} WHERE order_id = 1").show()

print("Sau MERGE (version", merge_version, "):")
spark.sql(f"SELECT * FROM bai12.orders_bronze VERSION AS OF {merge_version} WHERE order_id = 1").show()


**Đáp án Bài 4**: Không có lời giải cố định — đây là bài tập mở để tự luyện tổng hợp. Tiêu chí tự chấm: bảng có `PARTITIONED BY` hợp lý (cột cardinality thấp-vừa, hay filter), có ít nhất 1 `MERGE INTO` với cả `WHEN MATCHED` và `WHEN NOT MATCHED`, bảng có `TBLPROPERTIES (delta.enableChangeDataFeed = true)`, có chạy `OPTIMIZE` (có/không kèm `ZORDER BY`), và `DESCRIBE HISTORY` cho thấy rõ chuỗi operation từ tạo bảng → nạp dữ liệu → merge → optimize.

**Đáp án Bài 5 (gợi ý khung trả lời)**:
- **`OPTIMIZE`**: tần suất tỉ lệ nghịch với tần suất ghi (ghi càng thường xuyên/nhỏ lẻ → optimize càng cần chạy đều, ví dụ hàng ngày); có `ZORDER BY` theo cột cardinality cao hay xuất hiện trong `WHERE` mà không tiện làm partition (xem Bài 7).
- **`VACUUM`**: giữ retention mặc định 7 ngày trừ khi có lý do rõ ràng (môi trường test, chi phí storage là vấn đề nghiêm trọng) để hạ thấp hơn; luôn `DRY RUN` trước trên production (Bài 8).
- **CDF**: bật nếu có downstream cần đọc incremental changes (CDC ra hệ thống khác, đồng bộ analytics...); không cần nếu bảng chỉ được đọc full-scan định kỳ.
- **Partition/Z-order**: chọn dựa theo cardinality và tần suất xuất hiện trong filter, theo đúng nguyên tắc đã học ở Bài 3 và Bài 7.
